In [1]:
import cv2 
import mediapipe as mp
import numpy as np
import torch as pt


print("OpenCV:", cv2.__version__)
print("MediaPipe:", mp.__version__)
print("NumPy:", np.__version__)
print("PyTorch:", pt.__version__)

OpenCV: 4.13.0
MediaPipe: 0.10.35
NumPy: 2.4.4
PyTorch: 2.11.0+cu128


In [2]:
import axon

In [3]:
import torch.nn.functional as F

In [4]:
from torchvision import transforms

In [5]:
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

In [6]:
my_base_options = python.BaseOptions(
    model_asset_path='face_landmarker.task',
    delegate=python.BaseOptions.Delegate.CPU  # Running on the processor
)

In [7]:
options = vision.FaceLandmarkerOptions(
    base_options=my_base_options,  # <--- PLUGGED IN HERE
    num_faces=1
)

In [8]:
detector = vision.FaceLandmarker.create_from_options(options)

W0000 00:00:1781277766.588792   25525 face_landmarker_graph.cc:174] Sets FaceBlendshapesGraph acceleration to xnnpack by default.
I0000 00:00:1781277766.659894   25525 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1781277766.672636   25555 gl_context.cc:385] GL version: 3.2 (OpenGL ES 3.2 Mesa 26.0.3-1ubuntu1), renderer: Mesa Intel(R) Graphics (RPL-S)
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1781277766.684197   25529 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1781277766.707323   25545 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


In [9]:
def Right_resised_image(frame,result):
    if result==None:
        return None
    if result.face_landmarks is None or len(result.face_landmarks) == 0:
        return None
    Right_Eye_indices=[263,
    362,
    387,
    386,
    385,
    384,
    398,
    373,
    374,
    380,
    381,
    382]
    ih,iw,i_d=frame.shape
    X_coordinates_left=[int(result.face_landmarks[0][i].x *iw ) for i in Right_Eye_indices]
    Y_coordinate_left=[int(result.face_landmarks[0][i].y *ih ) for i in Right_Eye_indices]
    x_minl, x_maxl = min(X_coordinates_left), max(X_coordinates_left)
    y_minl, y_maxl = min(Y_coordinate_left), max(Y_coordinate_left)
    width_of_left_eye=x_maxl-x_minl
    hight_of_left_eye=y_maxl-y_minl
    x_minl -= 0.15 * width_of_left_eye
    x_maxl += 0.15 * width_of_left_eye
    y_minl -= 0.25 * hight_of_left_eye
    y_maxl += 0.25 * hight_of_left_eye
    x_minl = int(x_minl)
    x_maxl = int(x_maxl)
    y_minl = int(y_minl)
    y_maxl = int(y_maxl)
    x_minl = max(0, x_minl)
    y_minl = max(0, y_minl)
    x_maxl = min(iw, x_maxl)
    y_maxl = min(ih, y_maxl)
    left_eye_crop = frame[y_minl:y_maxl, x_minl:x_maxl]
    resized_left_eye=cv2.resize(left_eye_crop,(32,32))
    return resized_left_eye

In [10]:
def left_resised_image(frame,result):
    if result==None:
        return None
    if result.face_landmarks is None or len(result.face_landmarks) == 0:
        return None
    Left_Eye_indices=[33,
    133,
    160,
    159,
    158,
    157,
    173,
    144,
    145,
    153,
    154,
    155]
    ih,iw,i_d=frame.shape
    X_coordinates_left=[int(result.face_landmarks[0][i].x *iw ) for i in Left_Eye_indices]
    Y_coordinate_left=[int(result.face_landmarks[0][i].y *ih ) for i in Left_Eye_indices]
    x_minl, x_maxl = min(X_coordinates_left), max(X_coordinates_left)
    y_minl, y_maxl = min(Y_coordinate_left), max(Y_coordinate_left)
    width_of_left_eye=x_maxl-x_minl
    hight_of_left_eye=y_maxl-y_minl
    x_minl -= 0.15 * width_of_left_eye
    x_maxl += 0.15 * width_of_left_eye
    y_minl -= 0.25 * hight_of_left_eye
    y_maxl += 0.25 * hight_of_left_eye
    x_minl = int(x_minl)
    x_maxl = int(x_maxl)
    y_minl = int(y_minl)
    y_maxl = int(y_maxl)
    x_minl = max(0, x_minl)
    y_minl = max(0, y_minl)
    x_maxl = min(iw, x_maxl)
    y_maxl = min(ih, y_maxl)
    left_eye_crop = frame[y_minl:y_maxl, x_minl:x_maxl]
    resized_left_eye=cv2.resize(left_eye_crop,(32,32))
    return resized_left_eye

In [11]:
def combined_image(frame,result):
    if result==None:
        return None
    if result.face_landmarks is None or len(result.face_landmarks) == 0:
        return None
    left_eye=left_resised_image(frame,result)
    right_eye=Right_resised_image(frame,result)
    combined = np.hstack((left_eye, right_eye))
    return combined
    

In [12]:

device = pt.device("cuda" if pt.cuda.is_available() else "cpu")
print(f"Working on: {device}")

Working on: cuda


In [13]:
class Model:
    def __init__(self):
        self.layer_1 = axon.conv2D(1,8,device=device)
        self.layer_2 = axon.conv2D(8,16,device=device)
        self.layer_3 = axon.MLP([64,4],2048,device=device)

    def parameters(self):
        P = []
        P.extend(self.layer_1.parameters())
        P.extend(self.layer_2.parameters())
        P.extend(self.layer_3.parameters())
        return P

    def __call__(self, img):
        r1 = self.layer_1(img)
        r1 = pt.relu(r1)
        r1 = F.max_pool2d(r1,2)

        r2 = self.layer_2(r1)
        r2 = pt.relu(r2)
        r2 = F.max_pool2d(r2,2)

        y = r2.reshape(r2.shape[0], -1)

        r3 = self.layer_3(y)

        return r3
        

In [14]:
model=Model()

In [15]:
saved_params = pt.load("blink_model_weights.pt")

In [16]:
with pt.no_grad():
    for param, saved_param in zip(model.parameters(), saved_params):
        param.copy_(saved_param)

In [17]:
threshold=3
both_closed=0
right_closed=0
left_closed=0
cap = cv2.VideoCapture(0)
if not cap.isOpened():
    print("Cannot open camera")
    exit()
while True:
    ret, frame = cap.read()
 
    if not ret:
        print("Can't receive frame (stream end?). Exiting ...")
        break
    
    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)
    result = detector.detect(mp_image)
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    gray = gray[:,:,None]
    eyes=combined_image(gray,result)
    if eyes is not None:
        eyes_cuda=pt.tensor(eyes,device=device,dtype=pt.float32)
        eyes_cuda=eyes_cuda/255.0
        eyes_cuda=eyes_cuda.unsqueeze(0)
        eyes_cuda=eyes_cuda.unsqueeze(0)
        prediction=pt.argmax(model(eyes_cuda), dim=1).item()
        if prediction==0:
            both_closed=0
            right_closed=0
            left_closed=0
        if prediction==1:
            left_closed+=1
            if left_closed==threshold:
                print("Left_eye_blink_detected")
        if prediction==2:
            right_closed+=1
            if right_closed==threshold:
                print("Right_eye_blink_detected")
        if prediction==3 :
            both_closed+=1
            if both_closed==threshold:
                print("Full_blink_detected")
    
    
    cv2.imshow("frames",frame)
    if cv2.waitKey(1) == ord('q'):
        break
# When everything done, release the capture
cap.release()
cv2.destroyAllWindows()

QFontDatabase: Cannot find font directory /home/abhilash-a-kulkarni/miniconda3/envs/blink_project/lib/python3.11/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/abhilash-a-kulkarni/miniconda3/envs/blink_project/lib/python3.11/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/abhilash-a-kulkarni/miniconda3/envs/blink_project/lib/python3.11/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/abhilash-a-kulkarni/miniconda3/envs/blink_project/lib/python3.11/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://de

Left_eye_blink_detected
Right_eye_blink_detected
Left_eye_blink_detected
Right_eye_blink_detected
Full_blink_detected


In [18]:
cv2.imshow("eyes",eyes)
cv2.waitKey(0)
cv2.destroyAllWindows()

In [19]:
eyes.shape

(32, 64)

In [20]:
eyes_cuda.shape

torch.Size([1, 1, 32, 64])

In [21]:
gray.shape

(480, 640, 1)

In [22]:
eyes_cuda=eyes_cuda/255.0

In [23]:
eyes_cuda=eyes_cuda.unsqueeze(0)
eyes_cuda=eyes_cuda.unsqueeze(0)

In [24]:
eyes_cuda.shape

torch.Size([1, 1, 1, 1, 32, 64])

In [ ]:
type(prediction)